# Microsoft Agent Framework — Lab 4: Advanced Multi-Agent Workflows

This final lab demonstrates **complex multi-agent orchestration** using Workflows:

1. **Fan-out / Fan-in** — send one input to multiple agents in parallel, then collect results
2. **Debate workflow** — a Judge coordinates a Pros agent and a Cons agent, then decides
3. **Multiple output types** — routing different message types to different executors
4. **Putting it all together** — a full research pipeline

This mirrors the AutoGen Core distributed lab but without the gRPC overhead.  
The Workflow engine's superstep model handles parallel execution natively.

In [1]:
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [6]:
from agent_framework import Executor, WorkflowBuilder, WorkflowContext, handler
from agent_framework.openai import OpenAIChatCompletionClient

client = OpenAIChatCompletionClient(model="gpt-4o-mini")

---
## Part 1 — Fan-out / Fan-in Pattern

A **Splitter** sends the same input to both a **ProsAgent** and a **ConsAgent**.  
A **Merger** waits for both and combines their outputs.

```
input → Splitter → ProsAgent  ─┐
                  ConsAgent ─┤ → Merger → output
```

The Workflow engine runs ProsAgent and ConsAgent **concurrently** within a single superstep.

In [7]:
from dataclasses import dataclass


@dataclass
class ProsMessage:
    content: str


@dataclass
class ConsMessage:
    content: str


@dataclass
class MergedMessage:
    pros: str
    cons: str

In [15]:
class SplitterExecutor(Executor):
    """Fans out a topic to both the Pros and Cons executors."""

    def __init__(self):
        super().__init__(id="splitter")

    @handler
    async def split(self, topic: str, ctx: WorkflowContext[ProsMessage | ConsMessage]) -> None:
        await ctx.send_message(ProsMessage(content=topic))
        await ctx.send_message(ConsMessage(content=topic))


class ProsExecutor(Executor):
    """Researches the advantages of a topic."""

    def __init__(self):
        super().__init__(id="pros_agent")
        self._agent = client.as_agent(
            name="pros_agent",
            instructions=(
                "You research the PROS / advantages of a given topic. "
                "Respond with 3 concise bullet points."
            ),
        )

    @handler
    async def handle(self, msg: ProsMessage, ctx: WorkflowContext[str]) -> None:
        result = await self._agent.run(f"What are the pros of: {msg.content}")
        await ctx.send_message(f"PROS:\n{result}")


class ConsExecutor(Executor):
    """Researches the disadvantages of a topic."""

    def __init__(self):
        super().__init__(id="cons_agent")
        self._agent = client.as_agent(
            name="cons_agent",
            instructions=(
                "You research the CONS / disadvantages of a given topic. "
                "Respond with 3 concise bullet points."
            ),
        )

    @handler
    async def handle(self, msg: ConsMessage, ctx: WorkflowContext[str]) -> None:
        result = await self._agent.run(f"What are the cons of: {msg.content}")
        await ctx.send_message(f"CONS:\n{result}")


class MergerExecutor(Executor):
    """Collects all upstream results and yields the combined output."""

    def __init__(self):
        super().__init__(id="merger")
        self._results: list[str] = []

    @handler
    async def collect(self, text: str, ctx: WorkflowContext[str]) -> None:
        self._results.append(text)
        if len(self._results) >= 2:
            combined = "\n\n".join(self._results)
            await ctx.yield_output(combined)
            self._results.clear()

In [9]:
splitter = SplitterExecutor()
pros_exec = ProsExecutor()
cons_exec = ConsExecutor()
merger = MergerExecutor()

fan_builder = WorkflowBuilder(start_executor=splitter)
fan_builder.add_edge(splitter, pros_exec)
fan_builder.add_edge(splitter, cons_exec)
fan_builder.add_edge(pros_exec, merger)
fan_builder.add_edge(cons_exec, merger)
fan_workflow = fan_builder.build()

result = await fan_workflow.run("Using Microsoft Agent Framework for a new AI project")
display(Markdown(result.get_outputs()[0]))

PROS:
- **Modular and Scalable Architecture**: The Microsoft Agent Framework supports a modular structure, allowing developers to easily scale their AI projects and integrate various components as the project evolves.

- **Rich Development Tools**: It comes with a robust set of development tools and libraries, making it easier for developers to design, implement, and maintain AI solutions while ensuring high performance and flexibility.

- **Strong Community and Support**: Being part of the Microsoft ecosystem, users benefit from extensive documentation, community support, and resources which can aid in troubleshooting and enhance development efficiency.

CONS:
- **Limited Customization**: Microsoft Agent Framework may not offer the flexibility needed for specific AI requirements, potentially limiting the customization of machine learning models or user interactions.

- **Dependency on Aging Technology**: The framework has not seen significant updates or enhancements in recent years, raising concerns about long-term viability and compatibility with modern AI practices and technologies.

- **Narrow Community Support**: The ecosystem around Microsoft Agent Framework is smaller than those of more popular frameworks, which can lead to challenges in finding community resources, forums, and troubleshooting assistance.

---
## Part 2 — The Debate: Pros, Cons, and a Judge

We extend the fan-out pattern with a **JudgeExecutor** that reads the merged pros/cons and makes a final decision.

```
input → Splitter → ProsAgent ─┐
                  ConsAgent ─┤ → Merger → Judge → output
```

In [16]:
class JudgeExecutor(Executor):
    """Receives merged pros/cons and delivers a final verdict."""

    def __init__(self):
        super().__init__(id="judge")
        self._agent = client.as_agent(
            name="judge",
            instructions=(
                "You are a senior technology decision-maker. "
                "Given the pros and cons presented by your research team, "
                "deliver a clear YES or NO recommendation with a short rationale."
            ),
        )

    @handler
    async def decide(self, research: str, ctx: WorkflowContext[None, str]) -> None:
        verdict = await self._agent.run(
            f"Research findings:\n\n{research}\n\n"
            "Based purely on this research, should we proceed? Respond with YES or NO and explain why."
        )
        await ctx.yield_output(f"{research}\n\n## Decision\n\n{verdict}")

In [17]:
# Fresh instances (MergerExecutor is stateful so we need new ones)
splitter2 = SplitterExecutor()
pros_exec2 = ProsExecutor()
cons_exec2 = ConsExecutor()
merger2 = MergerExecutor()
judge = JudgeExecutor()

debate_builder = WorkflowBuilder(start_executor=splitter2)
debate_builder.add_edge(splitter2, pros_exec2)
debate_builder.add_edge(splitter2, cons_exec2)
debate_builder.add_edge(pros_exec2, merger2)
debate_builder.add_edge(cons_exec2, merger2)
debate_builder.add_edge(merger2, judge)
debate_workflow = debate_builder.build()

question = "Adopting Microsoft Agent Framework over AutoGen for our new agentic AI project"
result = await debate_workflow.run(question)
display(Markdown(result.get_outputs()[0]))

PROS:
- **Robust Ecosystem and Integration**: Microsoft Agent Framework provides seamless integration with existing Microsoft products and services, ensuring enhanced compatibility and easier deployment within established workflows.

- **Scalability and Flexibility**: The framework is built to accommodate growth, allowing developers to scale up functionalities and features as the project evolves, which can support complex applications and varied use cases.

- **Comprehensive Support and Documentation**: Adopting the Microsoft Agent Framework grants access to extensive resources, community support, and detailed documentation, which can streamline development processes and reduce time to market.

CONS:
- **Limited Flexibility**: Microsoft Agent Framework may impose constraints on customization and adaptability, making it harder to craft specific behaviors and responses tailored to unique project requirements compared to AutoGen's more flexible architecture.

- **Integration Complexity**: Integrating Microsoft Agent Framework with existing systems or third-party tools may create compatibility issues, leading to potential delays and increased development efforts as opposed to the potentially smoother integrations with AutoGen.

- **Resource Intensive**: The infrastructure and resources required to effectively implement and maintain the Microsoft Agent Framework can be higher, possibly leading to increased operational costs and escalated demands on technical support and infrastructure management compared to the more lightweight AutoGen solution.

---
## Part 3 — Streaming the Debate

Watch the workflow execute step by step with streaming events.

In [18]:
# Fresh instances again
splitter3 = SplitterExecutor()
pros_exec3 = ProsExecutor()
cons_exec3 = ConsExecutor()
merger3 = MergerExecutor()
judge3 = JudgeExecutor()

debate_builder3 = WorkflowBuilder(start_executor=splitter3)
debate_builder3.add_edge(splitter3, pros_exec3)
debate_builder3.add_edge(splitter3, cons_exec3)
debate_builder3.add_edge(pros_exec3, merger3)
debate_builder3.add_edge(cons_exec3, merger3)
debate_builder3.add_edge(merger3, judge3)
debate_workflow3 = debate_builder3.build()

question2 = "Using open-source LLMs instead of commercial models for production agents"

print(f"Running debate: {question2}\n")

async for event in debate_workflow3.run(question2, stream=True):
    if event.type == "executor_started":
        print(f"  → Starting: {event.executor_id}")
    elif event.type == "executor_completed":
        print(f"  ✓ Completed: {event.executor_id}")
    elif event.type == "output":
        print("\n=== FINAL OUTPUT ===")
        display(Markdown(event.data))

Running debate: Using open-source LLMs instead of commercial models for production agents

  ✓ Completed: splitter
  ✓ Completed: pros_agent
  ✓ Completed: cons_agent
  ✓ Completed: merger

=== FINAL OUTPUT ===


PROS:
- **Cost-Effectiveness**: Open-source LLMs are typically free to use, which can significantly reduce expenses associated with licensing fees required for commercial models, making them more accessible for startups and small businesses.

- **Customization and Flexibility**: Developers have the ability to modify and fine-tune open-source models to better suit their specific use cases, leading to potentially improved performance and alignment with unique business needs.

- **Community Support and Transparency**: Open-source LLMs benefit from a collaborative community that actively contributes to development, troubleshooting, and enhancements, offering diverse insights and ensuring greater transparency in model behavior compared to proprietary systems.

CONS:
- **Support and Reliability**: Open-source LLMs may lack the robust support and guaranteed uptime provided by commercial models, leading to potential challenges in troubleshooting and maintenance.

- **Performance and Quality Variability**: Open-source models can vary widely in performance and quality, leading to inconsistent results that may not meet the specific needs of production applications.

- **Security and Compliance Risks**: Using open-source software might expose organizations to vulnerabilities or compliance issues, as there may be fewer resources dedicated to addressing security concerns compared to commercial offerings.

  ✓ Completed: merger


---
## Part 4 — Full Research Pipeline

A complete end-to-end pipeline:
1. **TopicExpander** — breaks a broad question into 3 sub-questions
2. **Researcher** — answers each sub-question (fan-out × 3)
3. **Synthesizer** — merges all answers into a final report

This demonstrates dynamic fan-out where the number of branches depends on upstream output.

In [19]:
@dataclass
class ResearchQuestion:
    question: str
    index: int


@dataclass
class ResearchAnswer:
    question: str
    answer: str
    index: int

In [20]:
import json
from pydantic import BaseModel


class SubQuestions(BaseModel):
    questions: list[str]


class TopicExpanderExecutor(Executor):
    """Decomposes a broad topic into 3 focused research questions."""

    def __init__(self):
        super().__init__(id="topic_expander")
        self._agent = client.as_agent(
            name="topic_expander",
            instructions=(
                "You break a broad question into exactly 3 focused sub-questions for research. "
                "Return JSON: {\"questions\": [\"...\", \"...\", \"...\"]}"
            ),
        )

    @handler
    async def expand(self, topic: str, ctx: WorkflowContext[ResearchQuestion]) -> None:
        result = await self._agent.run(f"Break this into 3 sub-questions: {topic}")
        try:
            data = json.loads(str(result))
            questions = data.get("questions", [])
        except (json.JSONDecodeError, AttributeError):
            # Fallback: treat the whole output as one question
            questions = [str(result)]

        for i, q in enumerate(questions[:3]):
            await ctx.send_message(ResearchQuestion(question=q, index=i))


class ResearcherExecutor(Executor):
    """Answers a single research question concisely."""

    def __init__(self):
        super().__init__(id="researcher")
        self._agent = client.as_agent(
            name="researcher",
            instructions="You are a thorough research assistant. Answer in 2-3 sentences.",
        )

    @handler
    async def research(self, rq: ResearchQuestion, ctx: WorkflowContext[ResearchAnswer]) -> None:
        answer = await self._agent.run(rq.question)
        await ctx.send_message(ResearchAnswer(question=rq.question, answer=str(answer), index=rq.index))


class SynthesizerExecutor(Executor):
    """Collects all research answers and synthesizes a final report."""

    def __init__(self, expected_count: int = 3):
        super().__init__(id="synthesizer")
        self._expected = expected_count
        self._answers: list[ResearchAnswer] = []
        self._agent = client.as_agent(
            name="synthesizer",
            instructions=(
                "You are a senior analyst. "
                "Synthesize the research findings into a coherent, well-structured report. "
                "Use markdown formatting with headers."
            ),
        )

    @handler
    async def synthesize(self, answer: ResearchAnswer, ctx: WorkflowContext[None, str]) -> None:
        self._answers.append(answer)
        if len(self._answers) >= self._expected:
            self._answers.sort(key=lambda a: a.index)
            research_summary = "\n\n".join(
                f"**Q: {a.question}**\n{a.answer}" for a in self._answers
            )
            report = await self._agent.run(
                f"Synthesize these research findings into a report:\n\n{research_summary}"
            )
            await ctx.yield_output(str(report))
            self._answers.clear()

In [21]:
expander = TopicExpanderExecutor()
researcher = ResearcherExecutor()
synthesizer = SynthesizerExecutor(expected_count=3)

pipeline_builder = WorkflowBuilder(start_executor=expander)
pipeline_builder.add_edge(expander, researcher)
pipeline_builder.add_edge(researcher, synthesizer)
research_pipeline = pipeline_builder.build()

topic = "The future of multi-agent AI systems in enterprise software"
print(f"Research topic: {topic}\n")

result = await research_pipeline.run(topic)
display(Markdown(result.get_outputs()[0]))

Research topic: The future of multi-agent AI systems in enterprise software



# Report on the Integration of Multi-Agent AI Systems in Enterprise Software

## Introduction
The integration of multi-agent AI (Artificial Intelligence) systems into enterprise software represents a significant technological advancement with the potential to reshape operations within organizations. This report synthesizes findings related to the benefits, challenges, and implications for employee roles associated with adopting these systems.

## Benefits of Multi-Agent AI Systems

### 1. Enhanced Efficiency
Multi-agent AI systems facilitate autonomous decision-making and task allocation, leading to significantly streamlined workflows. By automating routine processes, these systems enable employees to focus on more strategic tasks, thereby increasing overall productivity.

### 2. Improved Adaptability
Organizations equipped with multi-agent systems can respond dynamically to evolving market conditions and environmental changes. Such adaptability enhances resource management and promotes faster problem-solving, which is crucial in today’s fast-paced business landscape.

### 3. Facilitated Collaboration
These systems support increased collaboration across different departments by improving communication channels. As agents work together, they can share insights and resources, driving innovation and fostering a culture of teamwork within the organization.

## Challenges of Implementing Multi-Agent AI Systems

### 1. Integration Issues
One of the primary challenges organizations face is the integration of multi-agent systems with existing software infrastructures. This may necessitate substantial modifications, redevelopment efforts, or even complete overhauls of legacy systems.

### 2. Interoperability Complexities
Ensuring seamless communication between diverse agents and existing systems can be complicated. Organizations must navigate the interoperability challenges that arise due to differences in technology standards and data formats.

### 3. Data Management and Security
The implementation of multi-agent systems often leads to an increase in data flow. Organizations must develop robust strategies for managing this influx while also safeguarding sensitive information to mitigate security risks.

### 4. Cultural Resistance
There is often resistance to adopting new technologies within the workforce. Organizations may encounter pushback from employees who are apprehensive about changes to their established workflows and roles.

### 5. Skill Requirements
The effective management and maintenance of multi-agent AI systems demand a specific skill set that may not currently exist within the organization. This necessitates investments in training or hiring skilled personnel, which can strain resources.

## Changes in Employee Roles and Responsibilities

### 1. Shift Towards Collaborative Roles
As routine tasks are increasingly handled by AI systems, employee roles are likely to shift towards more collaborative and strategic functions. Employees may find themselves focusing more on decision-making and working in teams to leverage AI insights.

### 2. Emphasis on Ethical AI Management
With the rise of AI-driven processes, employees will need to ensure that these technologies are used ethically. This may require a new layer of responsibility where individuals safeguard against biases in AI systems and maintain ethical guidelines.

### 3. Development of Technical Skills
There will be an increased need for employees to enhance their technical and analytical capabilities. Understanding AI systems and interpreting the data they generate will become critical skills for the modern workforce.

### 4. Fostering Innovation
The integration of multi-agent AI is poised to create an environment that prioritizes human creativity and complex problem-solving, encouraging innovation. Employees may find themselves in roles that require agility in thinking and adaptability to new ideas.

## Conclusion
The integration of multi-agent AI systems into enterprise software offers numerous potential benefits such as enhanced efficiency, improved adaptability, and increased collaboration. However, organizations must be prepared to tackle various implementation challenges, including integration issues, interoperability, data management, cultural resistance, and skill gaps. As employee roles evolve, there will be a greater emphasis on collaboration, ethical management, and technical skills, ultimately leading to a more innovative workplace environment. Organizations that navigate these dynamics successfully stand poised to leverage multi-agent AI systems for a competitive advantage.

---
## Part 5 — Comparing to AutoGen Distributed

The AutoGen distributed lab used **gRPC workers** (`GrpcWorkerAgentRuntime`) running as separate processes.  
Microsoft Agent Framework achieves the same result with the **Workflow engine** — no gRPC needed.

| AutoGen Distributed | Microsoft Agent Framework |
|---|---|
| `GrpcWorkerAgentRuntimeHost` | Built into Workflow engine |
| `GrpcWorkerAgentRuntime` × N | `Executor` classes |
| `RoutedAgent` + `@message_handler` | `Executor` + `@handler` |
| `SingleThreadedAgentRuntime` | `WorkflowBuilder.build()` |
| `await runtime.send_message(msg, AgentId(...))` | `await ctx.send_message(msg)` + edges |
| Manual gRPC setup | Zero infrastructure |

The Workflow engine runs executors **concurrently within each superstep** (Pregel model),  
providing deterministic execution without the complexity of distributed gRPC workers.

In [22]:
# Bonus: run the full debate pipeline with streaming to see all events
splitter_b = SplitterExecutor()
pros_b = ProsExecutor()
cons_b = ConsExecutor()
merger_b = MergerExecutor()
judge_b = JudgeExecutor()

final_builder = WorkflowBuilder(start_executor=splitter_b)
final_builder.add_edge(splitter_b, pros_b)
final_builder.add_edge(splitter_b, cons_b)
final_builder.add_edge(pros_b, merger_b)
final_builder.add_edge(cons_b, merger_b)
final_builder.add_edge(merger_b, judge_b)
final_workflow = final_builder.build()

final_question = "Should teams adopt Microsoft Agent Framework as the successor to AutoGen?"
print(f"Question: {final_question}\n")

async for event in final_workflow.run(final_question, stream=True):
    if event.type in ("executor_started", "executor_completed"):
        status = "▶" if event.type == "executor_started" else "✓"
        print(f"  {status} {event.executor_id}")
    elif event.type == "output":
        print()
        display(Markdown(event.data))

Question: Should teams adopt Microsoft Agent Framework as the successor to AutoGen?

  ✓ splitter
  ✓ pros_agent
  ✓ cons_agent
  ✓ merger



PROS:
- **Enhanced Customization:** Microsoft Agent Framework offers a more flexible and customizable environment for developing intelligent agents, allowing teams to tailor functionalities to specific business needs and workflows.

- **Improved Integration:** This framework provides better integration with existing Microsoft tools and services, streamlining workflows for teams already using the Microsoft ecosystem and enhancing collaboration across platforms.

- **Robust Community and Support:** As part of a widely adopted Microsoft technology, teams can benefit from extensive documentation, community support, and regular updates, ensuring access to resources and best practices for effective implementation.

CONS:
- **Complexity and Learning Curve**: Transitioning to the Microsoft Agent Framework can be challenging due to its complexity, requiring teams to invest significant time and resources in training and adapting to the new tools and frameworks.

- **Integration Issues**: Existing systems and applications may face compatibility issues when integrating with the Microsoft Agent Framework, potentially leading to costly delays and the need for substantial refactoring of current projects.

- **Vendor Lock-In**: Relying on the Microsoft ecosystem can create dependency on specific technologies and services, limiting flexibility and potentially increasing costs in the long term due to licensing and ongoing support requirements.

  ✓ merger


---
### Course Summary — Microsoft Agent Framework

Over these 4 labs you've covered the full Microsoft Agent Framework stack:

| Lab | Topics |
|---|---|
| **Lab 1** | Model clients, agents, tools, streaming |
| **Lab 2** | Structured outputs, sessions, multi-modal, multi-agent hand-off, MCP |
| **Lab 3** | Executors, WorkflowBuilder, agents-as-executors, workflow streaming |
| **Lab 4** | Fan-out/fan-in, parallel executors, dynamic workflows, full pipeline |

**Key takeaway:** Microsoft Agent Framework gives you:
- AutoGen's *simple agent abstractions* (`.as_agent()`, tools, sessions)
- Semantic Kernel's *enterprise features* (type safety, middleware, telemetry)
- A *graph-based workflow engine* for deterministic multi-agent orchestration

All with zero infrastructure — no gRPC, no message brokers, no separate runtime processes.